# SLD pipeline on Kaggle: component classifier, then full-sheet detection and graph recovery

One notebook, two stages, both models kept under `/kaggle/working/sld_pipeline/models` so a later session can reuse them.

**Stage A – component classifier** (`src/Object Detection/Component Training/`): 66 symbol types on isolated crops. Scores so far: 99.8 % val, 93 % on the unseen-style printed test set.

**Stage B – full sheets** (`src/Object Detection/SLD Training/`): YOLOv8 finds symbols, junctions and text on a whole drawing; `wires.py` traces the wires into nets and edges; the Stage A classifier re-labels every detected symbol crop.

**Before running**: GPU on, Internet on, attach three datasets: component crops `anayedeshan/sld-component-dataset` (`manifest.jsonl`, `images/train`), full sheets `anayedeshan/sld-full-sheets` (`data.yaml`, `images/`, `labels/`, `graphs/`), and printed test crops `anayedeshan/sld-computer-made-test`. From this PC, `python src/kaggle_runner/run.py sld-pipeline` pushes this notebook and attaches those three. Every cell is safe to re-run after a restart.

In [ ]:
# imports
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import Image as IPyImage, display

## Paths

In [ ]:
REPO_URL = "https://github.com/ishuu19/Single-Line-Diagram-and-Energy-Mapping.git"
REPO = Path("/kaggle/working/Single-Line-Diagram-and-Energy-Mapping")
COMP_DIR = REPO / "src" / "Object Detection" / "Component Training"
SLD_DIR = REPO / "src" / "Object Detection" / "SLD Training"
SAVE_DIR = Path("/kaggle/working/sld_pipeline")      # survives "Save Version": models + outputs of both stages
INPUT = Path("/kaggle/input")

COMP_EPOCHS = 0      # Stage A loads models/custom_best.keras from the cloned repo. Set >0 to train again.
SLD_EPOCHS = 60      # Stage B; 2 for a smoke run. 0 = skip training and reuse saved weights

## Helpers

In [ ]:
def sh(*cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    return (r.stdout + r.stderr).strip()


def find_dir(marker):
    """First folder under /kaggle/input that contains `marker` (e.g. images/train or data.yaml)."""
    for hit in INPUT.rglob(marker.split("/")[-1]):
        root = hit
        for _ in marker.split("/"):
            root = root.parent
        if (root / marker).exists():
            return root
    return None


def mounted(slug):
    """Attached dataset root. Kaggle uses either of these mount points."""
    name = slug.split("/")[-1]
    for c in (INPUT / "datasets" / slug, INPUT / name):
        if c.exists():
            return c
    return None


def find_tree(marker, under=None):
    """Directory that contains `marker`, under one dataset or all of /kaggle/input."""
    base = under or INPUT
    if not base.exists():
        return None
    for hit in base.rglob(marker.split("/")[-1]):
        root = hit
        for _ in marker.split("/"):
            root = root.parent
        if (root / marker).exists():
            return root
    return None


def link(src, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink() or dest.exists():
        dest.unlink() if dest.is_symlink() else shutil.rmtree(dest)
    os.symlink(src, dest, target_is_directory=True)


def restore(saved_sub, dest, pattern):
    """Copy models saved by an earlier session back into the repo folder."""
    src = SAVE_DIR / saved_sub
    if src.exists():
        dest.mkdir(parents=True, exist_ok=True)
        for f in src.glob(pattern):
            if not (dest / f.name).exists():
                shutil.copy(f, dest / f.name)


def keep(src, saved_sub):
    if src.exists():
        shutil.copytree(src, SAVE_DIR / saved_sub, dirs_exist_ok=True)

## Repo

In [ ]:
if (REPO / ".git").is_dir():
    sh("git", "pull", "--ff-only", cwd=REPO)
else:
    sh("git", "clone", "-q", REPO_URL, str(REPO))
assert (COMP_DIR / "train.py").exists() and (SLD_DIR / "train_sld.py").exists()

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras-tuner", "ultralytics", "scikit-image", "networkx"], check=True);

# Stage A – component classifier
Scripts are run as subprocesses so Stage A's `config.py` never collides with Stage B's.

In [ ]:
def crop_root():
    """Folder that has both manifest.jsonl and images/train.

    The sheet set also has images/train, so that marker alone picks the wrong dataset.
    Crops are nested: sld-component-dataset/component-symbols/.
    """
    bases = [
        INPUT / "datasets" / "anayedeshan" / "sld-component-dataset",
        INPUT / "sld-component-dataset",
        INPUT / "datasets" / "sld-component-dataset",
    ]
    bases.extend(p for p in INPUT.iterdir() if p.is_dir() and "component" in p.name)
    seen = set()
    for base in bases:
        if not base.is_dir() or base in seen:
            continue
        seen.add(base)
        for root in (base, base / "component-symbols"):
            if (root / "manifest.jsonl").is_file() and (root / "images" / "train").is_dir():
                return root
    return None

CROPS = crop_root()
assert CROPS is not None, (
    "component crops not found. /kaggle/input contains: "
    + ", ".join(p.name for p in INPUT.iterdir())
)
print(CROPS)
link(CROPS, REPO / "Data" / "component-symbols")

In [ ]:
# printed held-out test crops: attached dataset, else the repo zip
printed = REPO / "Data" / "component-symbols-test-printed"
if not (printed / "images" / "test").is_dir():
    test_ds = mounted("anayedeshan/sld-computer-made-test")
    src = None
    if test_ds:
        src = test_ds if (test_ds / "images" / "test").is_dir() else find_tree("images/test", test_ds)
    if src:
        link(src, printed)
    else:
        zip_path = REPO / "Data" / "component-symbols-test-printed.zip"
        if zip_path.exists():
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(REPO / "Data" if z.namelist()[0].startswith("component-symbols-test-printed/") else printed)

In [ ]:
restore("models/component", COMP_DIR / "models", "*.keras")
restore("models/component", COMP_DIR / "models", "*.json")
sorted(p.name for p in (COMP_DIR / "models").glob("*"))

### Data check

In [ ]:
print(sh(sys.executable, "-c", "from data import load_datasets; tr, va, cn, _ = load_datasets(); print(len(cn), 'classes |', int(tr.cardinality()), 'train batches |', int(va.cardinality()), 'val batches')", cwd=COMP_DIR).splitlines()[-1])

### Train

In [ ]:
if COMP_EPOCHS:
    log = sh(sys.executable, "train.py", "--epochs", str(COMP_EPOCHS), cwd=COMP_DIR)
    print("\n".join(l for l in log.splitlines() if "val_accuracy" in l)[-600:])
else:
    ready = COMP_DIR / "models" / "custom_best.keras"
    assert ready.is_file(), f"trained classifier missing from the repo: {ready}"
    print("Stage A skipped, using", ready)

### Evaluate

In [ ]:
have = {p.stem for p in (COMP_DIR / "models").glob("*.keras")}
COMP_TAG = next(t for t in ("custom_best", "custom", "notebook_best") if t in have)
COMP_TAG

In [ ]:
for split in ("val", "printed-test"):
    print(split, sh(sys.executable, "evaluate.py", "--model", COMP_TAG, "--split", split, cwd=COMP_DIR).splitlines()[:5])

In [ ]:
display(IPyImage(str(COMP_DIR / "outputs" / f"{COMP_TAG}_printed_test_confusion_matrix.png"), width=700))

In [ ]:
keep(COMP_DIR / "models", "models/component")
keep(COMP_DIR / "outputs", "outputs/component")

# Stage B – full sheets

In [ ]:
os.chdir(SLD_DIR)
sys.path.insert(0, str(SLD_DIR))
import config
from sld_data import load_classes, sheet_ids, image_path, load_graph, gt_detections
from wires import trace
from graph_eval import score_sheet, summarise, oracle_prediction
from train_sld import train
from predict_graph import load_detector, predict_sheet
from reclassify import load_classifier, reclassify
from viz import show_graph

In [ ]:
sheets_ds = mounted("anayedeshan/sld-full-sheets")
SHEETS = None
if sheets_ds:
    for c in (sheets_ds / "sld-sheets", sheets_ds):
        if (c / "data.yaml").exists() and (c / "graphs").is_dir():
            SHEETS = c
            break
    if SHEETS is None:
        hit = find_tree("data.yaml", sheets_ds)
        if hit and (hit / "graphs").is_dir():
            SHEETS = hit
if SHEETS is None:
    hit = find_tree("data.yaml")
    if hit and (hit / "graphs").is_dir():
        SHEETS = hit
assert SHEETS, "sld-sheets not found (anayedeshan/sld-full-sheets)"
link(SHEETS, config.SHEETS_DIR)
classes = load_classes()
len(classes), {s: len(sheet_ids(s)) for s in ("train", "val", "test")}

In [ ]:
restore("models/sld", config.MODEL_DIR, "*.pt")

### Data check

In [ ]:
sid = sheet_ids("val")[0]
image = Image.open(image_path(sid, "val"))
show_graph(image, {"symbols": gt_detections(load_graph(sid), classes), "edges": []}, f"{sid}: ground-truth boxes");

### Wire tracer with ground-truth boxes

In [ ]:
summarise([score_sheet(*oracle_prediction(s, "val", classes)) for s in sheet_ids("val")[:20]])

In [ ]:
graph, result = oracle_prediction(sid, "val", classes)
show_graph(image, result, f"{sid}: recovered graph (red power, blue measurement)");

### Train the detector

In [ ]:
from ultralytics.utils import LOGGER
if not getattr(LOGGER, "_sld_quiet", False):
    _info = LOGGER.info
    def _info_quiet(msg, *args, **kwargs):
        text = msg % args if args and isinstance(msg, str) else msg
        if not isinstance(text, str) or "duplicate labels removed" not in text:
            return _info(msg, *args, **kwargs)
        images = boxes = 0
        split = "scan"
        for line in text.splitlines():
            if "duplicate labels removed" not in line:
                continue
            images += 1
            plain = "".join(part.split("m", 1)[-1] for part in line.split("\x1b"))
            split = plain.split(":", 1)[0].strip() or split
            tail = plain.rsplit(":", 1)[-1].strip().split()
            if tail and tail[0].isdigit():
                boxes += int(tail[0])
        _info(f"{split}: {images} images, {boxes} duplicate labels removed")
    LOGGER.info = _info_quiet
    LOGGER._sld_quiet = True

if SLD_EPOCHS:
    WEIGHTS = train(epochs=SLD_EPOCHS, project=SAVE_DIR / "runs")
else:
    WEIGHTS = next(config.MODEL_DIR.glob("sld_*.pt"))
WEIGHTS

In [ ]:
if SLD_EPOCHS:
    display(Image.open(SAVE_DIR / "runs" / "sld" / "results.png"))

### Detector metrics on the test split

In [ ]:
from ultralytics import YOLO
from train_sld import resolve_data_yaml
data_yaml = resolve_data_yaml(config.DATA_YAML, SAVE_DIR / "data.yaml")
m = YOLO(str(WEIGHTS)).val(data=str(data_yaml), split="test", imgsz=config.IMG_SIZE, plots=False, verbose=False)
{"mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3)}

### Detector + tracer, then the Stage A classifier re-labels each symbol crop

In [ ]:
detector = load_detector(WEIGHTS)
classifier = load_classifier(COMP_DIR / "models" / f"{COMP_TAG}.keras")
test_ids = sheet_ids("test")

In [ ]:
def recover(sid, use_classifier=True):
    img = Image.open(image_path(sid, "test"))
    result, graph = predict_sheet(detector, img, classes, sid)
    if use_classifier:
        reclassify(img, result["symbols"], classifier)
    return img, result, graph

In [ ]:
rows_det = [score_sheet(load_graph(s), recover(s, use_classifier=False)[1]) for s in test_ids]
rows_clf = [score_sheet(load_graph(s), recover(s)[1]) for s in test_ids]
{"detector only": summarise(rows_det), "detector + classifier": summarise(rows_clf)}

In [ ]:
img, result, pred_graph = recover(test_ids[0])
show_graph(img, result, f"{test_ids[0]}: detected, traced, re-labelled");

In [ ]:
pred_graph["edges"][:5]

## Save both stages

In [ ]:
config.OUTPUT_DIR.mkdir(exist_ok=True)
(config.OUTPUT_DIR / "graph_pred_test.json").write_text(json.dumps({
    "detector": summarise(rows_det), "detector+classifier": summarise(rows_clf), "sheets": dict(zip(test_ids, rows_clf))}, indent=1))
keep(config.MODEL_DIR, "models/sld")
keep(config.OUTPUT_DIR, "outputs/sld")
sorted(str(p.relative_to(SAVE_DIR)) for p in (SAVE_DIR / "models").rglob("*") if p.is_file())

## Download to your PC (models and outputs only)

Bundles everything under `SAVE_DIR` (component + SLD models, metrics JSON, plots) and this notebook. **No** crop or sheet datasets. Click the link, or use Kaggle **Output** after **Save Version**. Unzip into `src/Object Detection/SLD Training/Results/` in the repo.

In [ ]:
BUNDLE_DIR = Path("/kaggle/working/sld_pipeline_artifacts")
BUNDLE_ZIP = Path("/kaggle/working/sld_pipeline_artifacts.zip")
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir()

In [ ]:
for sub in ("models", "outputs"):
    src = SAVE_DIR / sub
    if src.exists():
        shutil.copytree(src, BUNDLE_DIR / sub)

In [ ]:
runs_sld = SAVE_DIR / "runs" / "sld"
if runs_sld.exists():
    shutil.copytree(runs_sld, BUNDLE_DIR / "runs" / "sld")

In [ ]:
nb_repo = SLD_DIR / "notebooks" / "kaggle_sld_pipeline.ipynb"
if nb_repo.exists():
    shutil.copy(nb_repo, BUNDLE_DIR / "kaggle_sld_pipeline.ipynb")
for nb in Path("/kaggle/working").glob("*.ipynb"):
    if "sld" in nb.stem.lower() and nb.name != "kaggle_sld_pipeline.ipynb":
        shutil.copy(nb, BUNDLE_DIR / nb.name)
        break

In [ ]:
if BUNDLE_ZIP.exists():
    BUNDLE_ZIP.unlink()
with zipfile.ZipFile(BUNDLE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in BUNDLE_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(BUNDLE_DIR))

In [ ]:
from IPython.display import FileLink, display

files = sorted(p.as_posix() for p in BUNDLE_DIR.rglob("*") if p.is_file())
print(f"{len(files)} files, {BUNDLE_ZIP.stat().st_size / 1e6:.1f} MB")
for line in files[:40]:
    print(" ", line)
if len(files) > 40:
    print(f"  ... and {len(files) - 40} more")
display(FileLink(str(BUNDLE_ZIP), result_html_suffix=" — download zip"))